t-Testing

In [15]:
import pandas as pd
import numpy as np
from scipy import stats

Load the Data

In [16]:
df = pd.read_csv('../../output/processed_data/05_filtered.csv')

Daily Abnormal Returns

In [17]:
df['AR'] = df['R_i'] - (df['alpha'] + df['beta'] * df['R_m'])

Event Window

In [18]:
windows = {
    'CAR(-5,-1)': (-5, -1),
    'CAR(0,0)': (0, 0),
    'CAR(0,+5)': (0, 5),
    'CAR(-5,+30)': (-5, 30)
}

t-test Function

In [19]:
def run_cross_sectional_ttest(data, start_day, end_day, window_name):
    #Filter the data
    window_data = data[(data['day'] >= start_day) & (data['day'] <= end_day)]

    #sum daily ARto get CAR
    stock_cars = window_data.groupby(['symbol', 'sector'])['AR'].sum().reset_index()
    stock_cars.rename(columns={'AR': 'CAR'}, inplace=True)

    results = []

    # Sector level t-test
    sectors = stock_cars['sector'].unique()

    for sector in sectors:
        sector_data = stock_cars[stock_cars['sector'] == sector]['CAR']
        n_stocks = len(sector_data)
        
        if n_stocks > 1:
            mean_car = sector_data.mean()
            std_car = sector_data.std()
            t_stat, p_val = stats.ttest_1samp(sector_data, popmean=0)
        else:
            mean_car = sector_data.mean()
            std_car = np.nan
            t_stat = np.nan
            p_val = np.nan

        results.append({
            'Window': window_name,
            'Level': 'Sector',
            'Name': sector,
            'N': n_stocks,
            'Mean_CAR(%)': mean_car * 100, # Convert to percentage for readability
            'Std_Dev(%)': std_car * 100,
            't-statistic': t_stat,
            'p-value': p_val,
            'Significant_at_5%': p_val < 0.05 if pd.notna(p_val) else False
        })

    #Market t-test
    all_cars = stock_cars['CAR']
    n_total = len(all_cars)
    t_stat_mkt, p_val_mkt = stats.ttest_1samp(all_cars, popmean=0)

    results.append({
        'Window': window_name,
        'Level': 'Market',
        'Name': 'ALL STOCKS',
        'N': n_total,
        'Mean_CAR(%)': all_cars.mean() * 100, 
        'Std_Dev(%)': all_cars.std() * 100,
        't-statistic': t_stat_mkt,
        'p-value': p_val_mkt,
        'Significant_at_5%': p_val_mkt < 0.05 
    })

    return pd.DataFrame(results)



Test all windows

In [20]:
all_results = []

for window_name, (start, end) in windows.items():
    window_results_df = run_cross_sectional_ttest(df, start, end, window_name)
    all_results.append(window_results_df)

final_ttest_results = pd.concat(all_results,ignore_index=True)

formatted_results = final_ttest_results.copy()
formatted_results['Mean_CAR(%)'] = formatted_results['Mean_CAR(%)'].round(2)
formatted_results['Std_Dev(%)'] = formatted_results['Std_Dev(%)'].round(2)
formatted_results['t-statistic'] = formatted_results['t-statistic'].round(3)
formatted_results['p-value'] = formatted_results['p-value'].round(4)

Results

In [21]:
display_df = formatted_results[formatted_results['Window'] == 'CAR(-5,+30)'].sort_values(by='Mean_CAR(%)')
print(display_df[['Name', 'N', 'Mean_CAR(%)', 't-statistic', 'p-value', 'Significant_at_5%']])


formatted_results.to_csv('../../output/tables/ttest_results.csv', index=False)

                          Name    N  Mean_CAR(%)  t-statistic  p-value  \
77              Transportation    2       -18.65      -14.111   0.0450   
72                 Real Estate   16        -9.23       -2.352   0.0327   
79         Software & Services    1        -7.41          NaN      NaN   
80                   Utilities    5        -6.69       -1.829   0.1414   
64                   Insurance   11        -6.36       -2.803   0.0187   
63      Diversified Financials   33        -6.22       -3.068   0.0044   
78  Telecommunication Services    2        -5.34       -1.711   0.3368   
66                       Banks   10        -4.98       -1.678   0.1277   
75         Commercial Services    4        -4.67       -1.534   0.2226   
76              Food Retailing    1        -4.66          NaN      NaN   
83                  ALL STOCKS  221        -4.48       -6.033   0.0000   
69           Consumer Services   30        -4.26       -2.190   0.0367   
68             Food & Beverage   35   